# EXP007_GPU_V1 — engineering amendment E01

This corrects the missing `import math` in result serialization. **Scientific design unchanged.**
Stop the old notebook. Open this notebook on the same GPU model, select **Run all**,
authorise Drive, and upload `EXP007_GPU_V1_E01_input_bundle.zip` when prompted.

The original `NASA_EXP007_GPU_V1/results/` and its freeze remain untouched.
This notebook verifies that the failed original run has **zero committed fits**,
verifies the one-line source change, and requires the original environment fingerprint.
It records a provenance link and uses the separate `results_E01/` execution directory.
The first fit must train again because the original error preceded checkpoint commit.

No scientific settings, inputs, seeds, selection rules, fit budget (1,116), or tolerances change.
After interruption, use this E01 notebook again; verified E01 fits resume normally.
Do not use the old notebook again or run two notebooks concurrently.
If an original committed fit or an environment mismatch is found, the notebook stops
without modifying the original run. Do not delete or bypass these guards.


In [ ]:
# GPU + durable storage + immutable input (only initial Drive/upload interaction).
import hashlib, json, os, shutil, subprocess, sys, time, zipfile
from pathlib import Path, PurePosixPath
from google.colab import drive, files

EXPECTED_BUNDLE_SHA256 = "f620a98da63963f28f2ae6dc50c23baba5ca3b5dd9dcdb92fbab546a18f3b75a"
BUNDLE_NAME = "EXP007_GPU_V1_E01_input_bundle.zip"
if sys.version_info[:2] not in ((3, 12), (3, 13)):
    raise RuntimeError("This pinned environment supports Python 3.12/3.13. Do not change dependency versions silently.")
subprocess.run(["nvidia-smi", "--query-gpu=name,driver_version", "--format=csv"], check=True)
drive.mount("/content/drive")
BASE = Path("/content/drive/MyDrive/NASA_EXP007_GPU_V1")
BASE.mkdir(parents=True, exist_ok=True)
DURABLE_BUNDLE = BASE / BUNDLE_NAME

for run_dir in (BASE / "results", BASE / "results_E01"):
    lease_path = run_dir / "ACTIVE_RUN.json"
    if lease_path.exists() and time.time() - json.loads(lease_path.read_text())["heartbeat"] < 600:
        raise RuntimeError("A run may still be active. Stop it; after an abandoned runtime wait ten minutes. Do not delete its lease.")

def file_sha(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024*1024), b""): h.update(block)
    return h.hexdigest()

if not DURABLE_BUNDLE.exists():
    print("Select the supplied " + BUNDLE_NAME)
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise RuntimeError("Supply exactly the one input bundle.")
    name, data = next(iter(uploaded.items()))
    if hashlib.sha256(data).hexdigest() != EXPECTED_BUNDLE_SHA256:
        raise RuntimeError("Input bundle checksum mismatch; no training started.")
    temporary = DURABLE_BUNDLE.with_suffix(".zip.uploading")
    with open(temporary, "wb") as f:
        f.write(data); f.flush(); os.fsync(f.fileno())
    if file_sha(temporary) != EXPECTED_BUNDLE_SHA256:
        raise RuntimeError("Drive copy verification failed")
    os.replace(temporary, DURABLE_BUNDLE)
    del uploaded, data
if file_sha(DURABLE_BUNDLE) != EXPECTED_BUNDLE_SHA256:
    raise RuntimeError("Saved input changed; refusing to continue.")

WORK = Path("/content/exp007_gpu_v1_e01_work")
ROOT = WORK / "package"
WORK.mkdir(parents=True, exist_ok=True)
local_zip = WORK / BUNDLE_NAME
shutil.copy2(DURABLE_BUNDLE, local_zip)
if file_sha(local_zip) != EXPECTED_BUNDLE_SHA256: raise RuntimeError("Local bundle copy failed")
if ROOT.exists(): shutil.rmtree(ROOT)  # Reconstruct only disposable immutable input, never results.
ROOT.mkdir()
with zipfile.ZipFile(local_zip) as z:
    names = z.namelist()
    if len(names) != len(set(names)): raise RuntimeError("Duplicate archive entries")
    manifest = json.loads(z.read("PACKAGE_MANIFEST.json"))
    if set(names) != set(manifest["files"]) | {"PACKAGE_MANIFEST.json"}:
        raise RuntimeError("Unexpected archive members")
    for info in z.infolist():
        p = PurePosixPath(info.filename)
        if p.is_absolute() or ".." in p.parts or "\\" in info.filename:
            raise RuntimeError("Unsafe archive path")
        if (info.external_attr >> 16) & 0o170000 == 0o120000:
            raise RuntimeError("Symlinks are not accepted")
        payload = z.read(info)
        if info.filename != "PACKAGE_MANIFEST.json" and hashlib.sha256(payload).hexdigest() != manifest["files"][info.filename]:
            raise RuntimeError("File checksum mismatch: " + info.filename)
        target = ROOT.joinpath(*p.parts); target.parent.mkdir(parents=True, exist_ok=True); target.write_bytes(payload)
OUTPUT = BASE / "results_E01"
OUTPUT.mkdir(exist_ok=True)
print("Verified immutable TRAIN-only input. Durable results:", OUTPUT)

In [ ]:
# Verify the exact engineering amendment against the original frozen execution.
PARENT = BASE / "results"
PARENT_BUNDLE = BASE / "EXP007_GPU_V1_input_bundle.zip"
if not (PARENT / "RUN_FREEZE.json").is_file():
    raise RuntimeError("Original RUN_FREEZE.json is required for this recovery notebook.")
if not PARENT_BUNDLE.is_file() or file_sha(PARENT_BUNDLE) != "64cbf77f299261468d30b327147ea539fa610b394d025b963f8ab939d904ca4c":
    raise RuntimeError("Original bundle is missing or changed; preserve the run and investigate.")
parent_freeze = json.loads((PARENT / "RUN_FREEZE.json").read_text())
committed = list((PARENT / "fits").rglob("*.done.json"))
if committed:
    raise RuntimeError("Original committed fits found. E01 refuses to discard or relabel them; obtain a reviewed migration amendment.")
with zipfile.ZipFile(PARENT_BUNDLE) as z:
    parent_manifest_bytes = z.read("PACKAGE_MANIFEST.json")
    parent_manifest = json.loads(parent_manifest_bytes)
    if parent_freeze["package_manifest_sha256"] != hashlib.sha256(parent_manifest_bytes).hexdigest():
        raise RuntimeError("Original freeze does not match the original bundle.")
    if parent_freeze["protocol_sha256"] != hashlib.sha256(z.read("PROTOCOL.md")).hexdigest():
        raise RuntimeError("Original protocol freeze mismatch.")
    old_source = z.read("src/execution.py")
    expected_source = old_source.replace(b"from pathlib import Path\n", b"import math\nfrom pathlib import Path\n", 1)
    if old_source == expected_source or (ROOT / "src/execution.py").read_bytes() != expected_source:
        raise RuntimeError("E01 is not the exact missing-import correction.")
    if set(parent_manifest["files"]) != set(manifest["files"]):
        raise RuntimeError("Package file inventory changed.")
    changed = [p for p in manifest["files"] if manifest["files"][p] != parent_manifest["files"][p]]
    if changed != ["src/execution.py"]:
        raise RuntimeError("Unexpected scientific/source change: " + str(changed))
    config = json.loads((ROOT / "configs/exp007.json").read_text())
    if json.loads((PARENT / "effective_config.json").read_text()) != config:
        raise RuntimeError("Original effective configuration differs.")
    if parent_freeze["experiment"] != config["experiment_id"]:
        raise RuntimeError("Scientific experiment identity differs.")
if not (BASE / "requirements-resolved.txt").is_file():
    raise RuntimeError("Original dependency lock is required; do not regenerate it.")
AMENDMENT = {
    "scientific_experiment": "EXP007_GPU_V1", "engineering_revision": "E01",
    "reason": "Missing math import in clean_json; failure occurred before first checkpoint commit",
    "source_change": "Add import math to src/execution.py; no other source changes",
    "old_execution_sha256": hashlib.sha256(old_source).hexdigest(),
    "new_execution_sha256": file_sha(ROOT / "src/execution.py"),
    "parent_bundle_sha256": file_sha(PARENT_BUNDLE),
    "amended_bundle_sha256": file_sha(DURABLE_BUNDLE),
    "parent_run_freeze_sha256": file_sha(PARENT / "RUN_FREEZE.json"),
    "parent_run_hash": hashlib.sha256(json.dumps(parent_freeze,sort_keys=True,separators=(",",":"),allow_nan=False).encode()).hexdigest(),
    "new_package_manifest_sha256": file_sha(ROOT / "PACKAGE_MANIFEST.json"),
    "protocol_sha256": file_sha(ROOT / "PROTOCOL.md"),
    "configuration_sha256": file_sha(ROOT / "configs/exp007.json"),
    "parent_committed_fits": 0, "checkpoints_migrated": 0,
    "failed_fit_action": "Rerun from its original seed; original trained weights were not committed",
    "parent_output": str(PARENT), "amended_output": str(OUTPUT),
    "parent_environment_match_required": True,
}
print("Verified: one import added; science unchanged; original committed fits = 0.")

# Install the exact dependency versions from the successfully completed A100 study.
# Uses an isolated subprocess environment, so the Colab kernel does not need restarting.
import venv
ENV = WORK / "venv"
PYTHON = ENV / "bin/python"
if not PYTHON.exists(): venv.EnvBuilder(with_pip=True).create(ENV)
LOCK = BASE / "requirements-resolved.txt"
pip = [str(PYTHON), "-m", "pip"]
requirements = LOCK if LOCK.exists() else ROOT / "provenance/EXP006_requirements-resolved.txt"
subprocess.run(pip + ["install", "--disable-pip-version-check", "-r", str(requirements),
                     "--extra-index-url", "https://download.pytorch.org/whl/cu130"], check=True)
subprocess.run(pip + ["check"], check=True)
resolved = subprocess.check_output(pip + ["freeze", "--all"], text=True)
if LOCK.exists():
    if sorted(LOCK.read_text().splitlines()) != sorted(resolved.splitlines()):
        raise RuntimeError("Dependency versions changed. Do not continue or delete the lock.")
else:
    temp = LOCK.with_suffix(".tmp"); temp.write_text(resolved); os.replace(temp, LOCK)
shutil.copy2(LOCK, OUTPUT / LOCK.name)
RUN_ENV = os.environ.copy()
RUN_ENV.update(PYTHONHASHSEED="17", CUBLAS_WORKSPACE_CONFIG=":4096:8",
               OMP_NUM_THREADS="1", MKL_NUM_THREADS="1", OPENBLAS_NUM_THREADS="1",
               PYTHONUNBUFFERED="1", PYTHONDONTWRITEBYTECODE="1")
print("Isolated environment installed. GPU execution checks run before research fits.")


# Verify actual imported serialization and exact parent CUDA fingerprint in a separate process.
probe_code = """
import sys, json
sys.path.insert(0, sys.argv[1])
from execution import clean_json, configure_cuda
from common import validate_package
from pathlib import Path
validate_package(Path(sys.argv[1]).parent)
payload = {'finite': 1.25, 'zero': 0.0, 'nested': [float('nan'), float('inf'), -float('inf'), {'x': 2.5}], 'integer': 7, 'boolean': True, 'none': None}
expected = {'finite': 1.25, 'zero': 0.0, 'nested': [None, None, None, {'x': 2.5}], 'integer': 7, 'boolean': True, 'none': None}
if clean_json(payload) != expected: raise RuntimeError('Serialization regression failed')
json.dumps(clean_json(payload), allow_nan=False)
print(json.dumps(configure_cuda(), sort_keys=True))
"""
probe_result = subprocess.run([str(PYTHON), "-c", probe_code, str(ROOT / "src")],
                              text=True, capture_output=True, env=RUN_ENV)
if probe_result.returncode:
    print(probe_result.stdout); print(probe_result.stderr)
    raise RuntimeError("E01 serialization/CUDA preflight failed; no research fits started.")
observed_environment = json.loads(probe_result.stdout)
if observed_environment != parent_freeze["environment"]:
    different = sorted(k for k in set(observed_environment) | set(parent_freeze["environment"])
                       if observed_environment.get(k) != parent_freeze["environment"].get(k))
    raise RuntimeError("Original environment changed: " + str(different) + ". No training started; do not relax the freeze.")
# Immutable amendment receipt. Never alter the original freeze or old checkpoints.
receipt_dir = OUTPUT / "engineering_E01"; receipt_dir.mkdir(exist_ok=True)
def save_immutable(path, payload):
    if path.exists():
        if path.read_bytes() != payload: raise RuntimeError("Amendment provenance differs: " + str(path))
    else:
        temp = path.with_suffix(path.suffix + ".tmp")
        with open(temp, "wb") as f:
            f.write(payload); f.flush(); os.fsync(f.fileno())
        os.replace(temp, path)
save_immutable(receipt_dir / "AMENDMENT.json", json.dumps(AMENDMENT, indent=2, sort_keys=True).encode())
save_immutable(receipt_dir / "PARENT_RUN_FREEZE.json", (PARENT / "RUN_FREEZE.json").read_bytes())
save_immutable(receipt_dir / "PARENT_PACKAGE_MANIFEST.json", parent_manifest_bytes)
save_immutable(receipt_dir / "PARENT_execution.py", old_source)
save_immutable(receipt_dir / "PARENT_effective_config.json", (PARENT / "effective_config.json").read_bytes())
print("E01 regression passed; environment exactly matches parent. Ready to resume scientific jobs.")


In [ ]:
!python -m pip install -q virtualenv
!rm -rf /content/exp007_gpu_v1_e01_work/venv
!python -m virtualenv /content/exp007_gpu_v1_e01_work/venv

In [ ]:
# Complete workflow; no manual shortlist choice or stage switching.
LOGS = OUTPUT / "logs"; LOGS.mkdir(exist_ok=True)
log_path = LOGS / (time.strftime("session_%Y%m%dT%H%M%S") + ".log")
command = [str(PYTHON), "-u", str(ROOT / "src/run_exp007.py"), "--output", str(OUTPUT)]
with open(log_path, "a", buffering=1) as log:
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, bufsize=1, env=RUN_ENV)
    try:
        for line in process.stdout:
            print(line, end=""); log.write(line); log.flush()
        returncode = process.wait()
    except BaseException:
        process.terminate()
        try: process.wait(timeout=30)
        except subprocess.TimeoutExpired: process.kill(); process.wait()
        raise
if returncode != 0:
    raise RuntimeError("Stopped safely. Completed fits remain on Drive. See " + str(log_path))
status = json.loads((OUTPUT / "STATUS.json").read_text())
if status.get("status") != "COMPLETE" or status.get("fits") != 1116:
    raise RuntimeError("Workflow did not pass final completeness checks.")
print((OUTPUT / "SUMMARY.md").read_text())


In [ ]:
# Small report download; full checkpoint export is already on Drive.
from IPython.display import display, Markdown
display(Markdown((OUTPUT / "SUMMARY.md").read_text()))
exports = OUTPUT / "exports"
for p in sorted(exports.glob("*.zip")):
    print(f"{p.name}: {p.stat().st_size / 1024**2:.1f} MiB — {p}")
print("Full results and checkpoints remain saved even if a browser download is blocked.")
try:
    files.download(str(exports / "EXP007_GPU_V1_summary.zip"))
except Exception as exc:
    print("Browser download unavailable; use the saved Drive export:", exc)


## Engineering provenance

E01 changes only the missing `math` import and the derived package manifest.
The scientific experiment remains EXP007_GPU_V1. The original freeze and logs stay
in `results/`; the corrected execution has its own freeze in `results_E01/`,
linked through `engineering_E01/AMENDMENT.json`. Existing hash and resume guards remain active.
No training-path equivalence tolerance is changed, and no architecture retuning is introduced.
